# State Objective Rebound-Weighted BC v1

Goal: follow up `state_objective_budget_usage_bc_warm_start_v1`, where global BC fit passed but rebound-slice fit failed. This notebook tests whether a simple rebound-weighted behavior-cloning loss fixes that failure without creating a new overfit path.

External basis:
- Qlib keeps this work inside a repeatable quantitative investment workflow: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- FinRL's train / validate / test framing is kept as the validation boundary for future PPO work: https://github.com/AI4Finance-Foundation/FinRL and https://arxiv.org/abs/2011.09607
- Behavior-cloning / demonstration pretraining is used only as a warm start, following the DQfD-style precedent: https://arxiv.org/abs/1704.03732
- Class imbalance handling is inspired by focal/class-balanced loss literature, but kept conservative here because rebound labels are extremely sparse: https://arxiv.org/abs/1708.02002 and https://arxiv.org/abs/1901.05555

Governance:
- This is a model-selection diagnostic, not runtime promotion.
- The target remains `relative_return >= +30pp`, `max_dd <= 15%`, and walk-forward validation after PPO.
- If simple rebound weighting cannot clear the rebound-slice gate, the next step should be target enrichment or state-conditioned training objective, not a larger weight grid.


In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime, timezone
import json
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "SPEC.md").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor,
    fit_bc_actor,
)
from src.soxl_quant.config import load_profile
from src.soxl_quant.policy.observation import observation_dim_for_profile
from src.soxl_quant.policy.policy_builder import (
    build_policy,
    build_state_objective_warm_start_dataset_from_frame,
)

EXPERIMENT = "state_objective_rebound_weighted_bc_v1"
CARD = ROOT / "runtime" / "model_selection" / "card" / EXPERIMENT
SOURCE_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_budget_usage_bc_warm_start_v1"
TEACHER_CARD = ROOT / "runtime" / "model_selection" / "card" / "state_objective_fold_native_teacher_router_v1"
TARGETS_PATH = TEACHER_CARD / "fold_native_teacher_targets.csv"
FRAME_PATH = ROOT / "data" / "soxl_profile_balanced_rule_residual_v22.pkl"
PROFILE_NAME = "risk_budget_usage_v1"
CARD.mkdir(parents=True, exist_ok=True)

source_selection = json.loads((SOURCE_CARD / "selection.json").read_text())
profile = load_profile(PROFILE_NAME)
frame = pd.read_pickle(FRAME_PATH)
targets = pd.read_csv(TARGETS_PATH, parse_dates=["datetime"])
dataset = build_state_objective_bc_dataset_from_frame(
    frame,
    targets=targets,
    profile=profile,
    sample_size=0,
)

metadata_summary = {
    "profile": PROFILE_NAME,
    "feature_frame": str(FRAME_PATH.relative_to(ROOT)),
    "source_decision": source_selection.get("decision"),
    "source_rebound_accuracy": source_selection.get("rebound_accuracy"),
    "samples": int(len(dataset.actions)),
    "metadata_rows": int(len(dataset.metadata)),
    "selected_rebound_rows": int(dataset.metadata["is_rebound_cluster"].sum()),
    "selected_downtrend_rows": int(dataset.metadata["is_downtrend_state"].sum()),
    "target_rows": int(len(targets)),
    "target_rebound_rows": int(targets["is_rebound_cluster"].astype(bool).sum()),
}
(CARD / "metadata_summary.json").write_text(json.dumps(metadata_summary, indent=2, sort_keys=True), encoding="utf-8")
metadata_summary

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{'profile': 'risk_budget_usage_v1',
 'feature_frame': 'data/soxl_profile_balanced_rule_residual_v22.pkl',
 'source_decision': 'diagnostic_bc_global_fit_rebound_slice_weak_no_promotion',
 'source_rebound_accuracy': 0.42857142857142855,
 'samples': 755,
 'metadata_rows': 755,
 'selected_rebound_rows': 4,
 'selected_downtrend_rows': 477,
 'target_rows': 1261,
 'target_rebound_rows': 7}

## Weight Screen

The screen holds the actor architecture, seed, learning rate, and training steps fixed. Only the rebound sample weight changes. This keeps the test narrow: if weighting helps, rebound accuracy should improve without degrading ordinary/downtrend state fit.


In [2]:
def eval_target_subset(actor, label: str, subset: pd.DataFrame) -> dict[str, object]:
    if subset.empty:
        return {"slice": label, "samples": 0, "loss": np.nan, "accuracy": np.nan}
    obs, actions = build_state_objective_warm_start_dataset_from_frame(
        frame,
        targets=subset,
        profile=profile,
        sample_size=0,
    )
    metrics = evaluate_bc_actor(actor, obs, actions)
    metrics["slice"] = label
    return metrics

ordinary_targets = targets[
    (~targets["is_rebound_cluster"].astype(bool))
    & (~targets["is_downtrend_state"].astype(bool))
]
state_subsets = {
    "all": targets,
    "rebound": targets[targets["is_rebound_cluster"].astype(bool)],
    "downtrend": targets[targets["is_downtrend_state"].astype(bool)],
    "ordinary": ordinary_targets,
}

screen_rows = []
state_rows = []
weight_grid = [1, 5, 8, 10, 12, 20, 40]
train_config = StateObjectiveBCTrainingConfig(steps=600, lr=0.001, batch_size=128, seed=42)
rebound_mask = dataset.metadata["is_rebound_cluster"].to_numpy(dtype=bool)

for rebound_weight in weight_grid:
    sample_weights = np.ones(len(dataset.actions), dtype=np.float32)
    if rebound_weight != 1:
        sample_weights[rebound_mask] *= float(rebound_weight)
    torch.manual_seed(int(train_config.seed))
    _policy, actor = build_policy(observation_dim_for_profile(profile), profile)
    fit_metrics = fit_bc_actor(
        actor,
        dataset.obs,
        dataset.actions,
        train_config,
        sample_weight_array=sample_weights if rebound_weight != 1 else None,
    )
    row = {
        "rebound_weight": float(rebound_weight),
        **fit_metrics,
    }
    for slice_name, subset in state_subsets.items():
        metrics = eval_target_subset(actor, slice_name, subset)
        row[f"{slice_name}_accuracy"] = float(metrics["accuracy"])
        row[f"{slice_name}_loss"] = float(metrics["loss"])
        row[f"{slice_name}_samples"] = int(metrics["samples"])
        state_rows.append({"rebound_weight": float(rebound_weight), **metrics})
    screen_rows.append(row)

screen = pd.DataFrame(screen_rows)
state_metrics = pd.DataFrame(state_rows)
# Gate-oriented score: rebound first, then protect global and ordinary fit.
screen["rebound_gap"] = (0.60 - screen["rebound_accuracy"]).clip(lower=0.0)
screen["global_gap"] = (0.80 - screen["all_accuracy"]).clip(lower=0.0)
screen["ordinary_gap"] = (0.65 - screen["ordinary_accuracy"]).clip(lower=0.0)
screen["weighted_bc_loss"] = (
    10.0 * screen["rebound_gap"]
    + 2.0 * screen["global_gap"]
    + 2.0 * screen["ordinary_gap"]
    - 0.1 * screen["downtrend_accuracy"]
)
screen = screen.sort_values(["weighted_bc_loss", "rebound_gap", "all_accuracy"], ascending=[True, True, False]).reset_index(drop=True)
screen.to_csv(CARD / "rebound_weight_screen.csv", index=False)
state_metrics.to_csv(CARD / "state_slice_metrics.csv", index=False)
screen

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


,rebound_weight,samples,steps,lr,batch_size,before_loss,before_accuracy,train_last_loss,train_last_batch_accuracy,after_loss,...,ordinary_accuracy,ordinary_loss,ordinary_samples,weight_mean,weight_min,weight_max,rebound_gap,global_gap,ordinary_gap,weighted_bc_loss
0,40.0,755,600,0.001,128,1.614131,0.129801,1.042207,0.859375,1.064427,...,0.708633,1.196148,278,1.206622,1.0,40.0,0.171429,0.0,0.0,1.627074
1,8.0,755,600,0.001,128,1.614131,0.129801,1.054354,0.851562,1.090178,...,0.694245,1.214581,278,1.037086,1.0,8.0,0.171429,0.0,0.0,1.629590
2,10.0,755,600,0.001,128,1.614131,0.129801,1.053977,0.851562,1.087101,...,0.697842,1.207293,278,1.047682,1.0,10.0,0.171429,0.0,0.0,1.629590
3,20.0,755,600,0.001,128,1.614131,0.129801,1.059560,0.843750,1.083809,...,0.708633,1.197848,278,1.100662,1.0,20.0,0.171429,0.0,0.0,1.629799
4,5.0,755,600,0.001,128,1.614131,0.129801,1.052908,0.851562,1.084966,...,0.697842,1.204300,278,1.021192,1.0,5.0,0.171429,0.0,0.0,1.629799
5,12.0,755,600,0.001,128,1.614131,0.129801,1.047232,0.859375,1.093055,...,0.697842,1.201410,278,1.058278,1.0,12.0,0.171429,0.0,0.0,1.630638
6,1.0,755,600,0.001,128,1.614131,0.129801,1.064738,0.843750,1.073260,...,0.723022,1.174270,278,NaN,NaN,NaN,0.171429,0.0,0.0,1.632944


In [3]:
state_metrics.pivot_table(
    index="rebound_weight",
    columns="slice",
    values="accuracy",
    aggfunc="first",
).reset_index()

slice,rebound_weight,all,downtrend,ordinary,rebound
0,1.0,0.831788,0.813417,0.723022,0.428571
1,5.0,0.818543,0.844864,0.697842,0.428571
2,8.0,0.818543,0.846960,0.694245,0.428571
3,10.0,0.818543,0.846960,0.697842,0.428571
4,12.0,0.811921,0.836478,0.697842,0.428571
5,20.0,0.821192,0.844864,0.708633,0.428571
6,40.0,0.842384,0.872117,0.708633,0.428571


## Decision

The pass gate is intentionally stricter than “best available”: rebound accuracy must clear 0.60 while global and ordinary-state fit remain acceptable. A near miss is not promoted into PPO because the source labels are sparse and derived from prior review evidence.


In [4]:
best = screen.iloc[0].to_dict()
full_gate = bool(
    best["rebound_accuracy"] >= 0.60
    and best["all_accuracy"] >= 0.80
    and best["ordinary_accuracy"] >= 0.65
)
source_rebound_accuracy = float(source_selection.get("rebound_accuracy", np.nan))
rebound_improvement = float(best["rebound_accuracy"] - source_rebound_accuracy)
near_miss = bool(
    best["rebound_accuracy"] >= 0.55
    and best["all_accuracy"] >= 0.80
    and best["ordinary_accuracy"] >= 0.65
)
if full_gate:
    decision = "rebound_weighting_passed_diagnostic_gate"
elif near_miss:
    decision = "rebound_weighting_near_miss_no_promotion"
elif rebound_improvement > 0.0:
    decision = "rebound_weighting_partial_improvement_no_promotion"
else:
    decision = "rebound_weighting_no_improvement_no_promotion"

selection = {
    "experiment": EXPERIMENT,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "decision": decision,
    "profile": PROFILE_NAME,
    "source_experiment": "state_objective_budget_usage_bc_warm_start_v1",
    "promotion_allowed": False,
    "promotion_blocker": "Rebound-weighted BC is not a backtest and did not clear the rebound-slice gate; no runtime or PPO handoff promotion from this screen.",
    "full_gate": full_gate,
    "near_miss": near_miss,
    "best_rebound_weight": float(best["rebound_weight"]),
    "best_rebound_accuracy": float(best["rebound_accuracy"]),
    "best_all_accuracy": float(best["all_accuracy"]),
    "best_downtrend_accuracy": float(best["downtrend_accuracy"]),
    "best_ordinary_accuracy": float(best["ordinary_accuracy"]),
    "source_rebound_accuracy": source_rebound_accuracy,
    "rebound_improvement": rebound_improvement,
    "target_rebound_accuracy_min": 0.60,
    "next_mechanism": "target_enrichment_or_state_conditioned_loss_before_walkforward_ppo" if not full_gate else "walkforward_ppo_from_rebound_weighted_bc_actor",
    "acceptance_gate_for_next_run": {
        "relative_return_min_pp": 30.0,
        "max_dd_max_pct": 15.0,
        "walkforward_required": True,
        "fixed_split_required": True,
        "rebound_slice_accuracy_min": 0.60,
        "target_enrichment_required_if_rebound_rows_below": 20,
    },
    "sidecars": [
        "metadata_summary.json",
        "rebound_weight_screen.csv",
        "state_slice_metrics.csv",
        "summary.json",
    ],
    "external_sources": [
        "https://github.com/microsoft/qlib",
        "https://arxiv.org/abs/2009.11189",
        "https://github.com/AI4Finance-Foundation/FinRL",
        "https://arxiv.org/abs/2011.09607",
        "https://arxiv.org/abs/1704.03732",
        "https://arxiv.org/abs/1708.02002",
        "https://arxiv.org/abs/1901.05555",
    ],
}
summary = {
    "conclusion": [
        f"Best rebound weight was {best['rebound_weight']:.0f}x with rebound accuracy {best['rebound_accuracy']:.3f}, versus source {selection['source_rebound_accuracy']:.3f}; improvement {selection['rebound_improvement']:.3f}.",
        f"Best all/downtrend/ordinary accuracies were {best['all_accuracy']:.3f}/{best['downtrend_accuracy']:.3f}/{best['ordinary_accuracy']:.3f}.",
        "The 0.60 rebound gate did not pass; deterministic weighting produced no rebound improvement, so it is not a PPO-ready warm start.",
        "Next step should enrich rebound targets or use state-conditioned loss/sampling before walk-forward PPO, rather than increasing the same weight grid.",
    ],
    "metadata_summary": metadata_summary,
    "best_row": best,
    "selection": selection,
}
(CARD / "selection.json").write_text(json.dumps(selection, indent=2, sort_keys=True), encoding="utf-8")
(CARD / "summary.json").write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
selection

{'experiment': 'state_objective_rebound_weighted_bc_v1',
 'created_at': '2026-05-25T12:22:26.043889+00:00',
 'decision': 'rebound_weighting_no_improvement_no_promotion',
 'profile': 'risk_budget_usage_v1',
 'source_experiment': 'state_objective_budget_usage_bc_warm_start_v1',
 'promotion_allowed': False,
 'promotion_blocker': 'Rebound-weighted BC is not a backtest and did not clear the rebound-slice gate; no runtime or PPO handoff promotion from this screen.',
 'full_gate': False,
 'near_miss': False,
 'best_rebound_weight': 40.0,
 'best_rebound_accuracy': 0.42857142857142855,
 'best_all_accuracy': 0.8423841059602649,
 'best_downtrend_accuracy': 0.8721174004192872,
 'best_ordinary_accuracy': 0.7086330935251799,
 'source_rebound_accuracy': 0.42857142857142855,
 'rebound_improvement': 0.0,
 'target_rebound_accuracy_min': 0.6,
 'next_mechanism': 'target_enrichment_or_state_conditioned_loss_before_walkforward_ppo',
 'acceptance_gate_for_next_run': {'relative_return_min_pp': 30.0,
  'max_dd